In [3]:
c = [1,2,3,4,5,6,7,8,9]
c[1:-1]

[2, 3, 4, 5, 6, 7, 8]

In [ ]:
def plot_img_from_latent(model, z, target, device):
    model.eval
    z_input = torch.tensor([z], dtype=torch.float).to(device)
    img = model.decode(z_input)
    img_dysplay = img.cpu().detach().numpy()
    if img_dysplay.ndim == 3 and img_dysplay.shape[0] == 1:
        img_dysplay = img_dysplay.squeeze(0)
    target.clear()
    target.imshow(img_dysplay, cmap='gray')
    z_coords_str = ", ".join([f"{c:.2f}" for c in z])
    target.set_title(f"Immagine da z=({z_coords_str})")
    target.axis('off')
    if hasattr(target.figure, 'canvas'):
        target.figure.canvas.draw_idle()

fig, (ax_scatter, ax_image_display) = plt.subplots(1, 2, figsize=(12, 5))
if hasattr(fig.canvas, 'header_visible'): # Per ipympl
    fig.canvas.header_visible = False
    fig.canvas.footer_visible = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
axis_x = []
axis_y = []
answers = []
with torch.no_grad():
    for img, labels in dataloader:
        model.eval()
        answers += labels.tolist()
        img = img.to(device)
        z = model(img)[-1]
        axis_x += z[:,0].tolist()
        axis_y += z[:,1].tolist()


scatter_plot = ax_scatter.scatter(axis_x, axis_y, c=answers, cmap='tab10', picker=5) # picker abilita il click
if hasattr(fig, 'colorbar'):
    fig.colorbar(scatter_plot, ax=ax_scatter, label='Classe')
ax_scatter.set_xlabel("Dimensione Latente 1 (z0)")
ax_scatter.set_ylabel("Dimensione Latente 2 (z1)")
ax_scatter.set_title("Spazio Latente (Clicca un punto)")

# 2. Placeholder per l'immagine generata
ax_image_display.set_title("Immagine Generata da Z")
ax_image_display.axis('off')
img_placeholder = np.zeros((28,28)) # Assumendo immagini MNIST 28x28
imshow_obj = ax_image_display.imshow(img_placeholder, cmap='gray')


# Widget per mostrare le coordinate (opzionale)
coord_label = widgets.Label()
display(coord_label)

# Funzione per gestire il click del mouse sullo scatter plot
def on_scatter_click(event):
    if event.mouseevent.inaxes == ax_scatter:
        # Per lo scatter plot, event.ind dà gli indici dei punti cliccati
        if len(event.ind) > 0:
            # Prendi il primo punto cliccato se ce ne sono multipli vicini
            idx = event.ind[0]
            z0_clicked = axis_x[idx]
            z1_clicked = axis_y[idx]
            
            clicked_z_coords_list = [z0_clicked, z1_clicked]
            
            coord_label.value = f'Z cliccato: ({z0_clicked:.2f}, {z1_clicked:.2f})'
            
            # Chiama la funzione per plottare l'immagine
            # 'model' e 'device' devono essere accessibili (definite globalmente o passate)
            plot_img_from_latent(model, clicked_z_coords_list, ax_image_display, device)
            
            # Aggiorna il display della figura (necessario per %matplotlib widget)
            event.canvas.draw_idle()
    else:
        coord_label.value = 'Z cliccato: Fuori dal grafico scatter'

# Collega l'evento di click del mouse (pick_event) allo scatter plot
fig.canvas.mpl_connect('motion_notify_event', on_scatter_click)

plt.tight_layout()
plt.show()